 RoBERTa Emotion Model





### Cell 1: Install Dependencies and Mount Google Drive

In [ ]:
!pip -q install -U transformers datasets scikit-learn accelerate matplotlib

from google.colab import drive
drive.mount('/content/drive')

### Cell 2: Configuration, Directories, and Seed

In [ ]:
import os, json, random, time
from pathlib import Path
import numpy as np
import torch

CFG = dict(
    seed=42,
    model_name="roberta-base",
    max_length=64,
    batch_size=32,
    eval_batch_size=128,
    lr=2e-5,
    weight_decay=0.01,
    epochs=5,
    warmup_ratio=0.1,
    patience=2,
    pos_weight_clip=10.0,
    out_dir="/content/drive/MyDrive/component3/roberta_emotion",
)

OUT = Path(CFG["out_dir"])
(OUT / "figures").mkdir(parents=True, exist_ok=True)

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "(no GPU! switch runtime to T4)")

### Cell 3: Load GoEmotions Dataset & Compute Imbalance Weights

In [ ]:
from datasets import load_dataset

raw = load_dataset("google-research-datasets/go_emotions", "simplified")
LABELS = raw["train"].features["labels"].feature.names      # 28 names, neutral is last
NUM_LABELS = len(LABELS)
print(NUM_LABELS, "labels:", LABELS)

def to_arrays(split):
    texts = list(split["text"])
    Y = np.zeros((len(texts), NUM_LABELS), dtype=np.float32)
    for i, lbls in enumerate(split["labels"]):
        Y[i, lbls] = 1.0
    return texts, Y

train_texts, Y_train = to_arrays(raw["train"])
val_texts,   Y_val   = to_arrays(raw["validation"])
test_texts,  Y_test  = to_arrays(raw["test"])
print(f"Dataset Splits: Train={len(train_texts):,}, Val={len(val_texts):,}, Test={len(test_texts):,}")

pos = Y_train.sum(0); neg = len(Y_train) - pos
POS_WEIGHT = torch.tensor(np.clip(neg / np.maximum(pos, 1), 1.0, CFG["pos_weight_clip"]), dtype=torch.float32)
print("pos_weight range:", POS_WEIGHT.min().item(), "to", POS_WEIGHT.max().item())

### Cell 4: Tokenization & PyTorch DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])

class EmoDataset(Dataset):
    def __init__(self, texts, Y):
        enc = tokenizer(texts, truncation=True, max_length=CFG["max_length"])
        self.ids, self.mask, self.Y = enc["input_ids"], enc["attention_mask"], Y
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        return {"input_ids": self.ids[i], "attention_mask": self.mask[i], "labels": torch.tensor(self.Y[i])}

def collate(batch):
    labels = torch.stack([b.pop("labels") for b in batch])
    padded = tokenizer.pad(batch, return_tensors="pt")
    padded["labels"] = labels
    return padded

g = torch.Generator(); g.manual_seed(CFG["seed"])
train_dl = DataLoader(EmoDataset(train_texts, Y_train), batch_size=CFG["batch_size"], shuffle=True, collate_fn=collate, generator=g)
val_dl   = DataLoader(EmoDataset(val_texts, Y_val),   batch_size=CFG["eval_batch_size"], collate_fn=collate)
test_dl  = DataLoader(EmoDataset(test_texts, Y_test), batch_size=CFG["eval_batch_size"], collate_fn=collate)

### Cell 5: Evaluation Metrics & Inference Helpers

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, f1_score

def compute_metrics(Y_true, Y_pred):
    out = {"subset_accuracy": float(accuracy_score(Y_true, Y_pred))}
    for avg in ["micro", "macro", "weighted"]:
        p, r, f, _ = precision_recall_fscore_support(Y_true, Y_pred, average=avg, zero_division=0)
        out[avg] = {"precision": float(p), "recall": float(r), "f1": float(f)}
    p, r, f, s = precision_recall_fscore_support(Y_true, Y_pred, average=None, zero_division=0)
    out["per_class"] = {LABELS[i]: {"precision": float(p[i]), "recall": float(r[i]),
                        "f1": float(f[i]), "support": int(s[i])} for i in range(NUM_LABELS)}
    return out

@torch.no_grad()
def get_probs(model, dl):
    model.eval(); chunks = []
    for b in dl:
        b = {k: v.to(device) for k, v in b.items() if k != "labels"}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            logits = model(**b).logits
        chunks.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(chunks)

### Cell 6: Model Training (Auto-Resumes from Checkpoint on Disconnect)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    CFG["model_name"], num_labels=NUM_LABELS, problem_type="multi_label_classification").to(device)

no_decay = ["bias", "LayerNorm.weight"]
params = [
    {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": CFG["weight_decay"]},
    {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
]
optimizer = torch.optim.AdamW(params, lr=CFG["lr"])
total_steps = len(train_dl) * CFG["epochs"]
scheduler = get_linear_schedule_with_warmup(optimizer, int(CFG["warmup_ratio"] * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT.to(device))

state = {"epoch": 0, "best_f1": -1.0, "best_epoch": 0, "bad": 0, "history": []}
CKPT, BEST_DIR = OUT / "last.pt", OUT / "best_model"
if CKPT.exists():
    ck = torch.load(CKPT, map_location=device)
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["opt"])
    scheduler.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"]); state = ck["state"]
    print(f"Resumed from epoch {state['epoch']}")

for epoch in range(state["epoch"], CFG["epochs"]):
    model.train(); t0, running = time.time(), 0.0
    for step, b in enumerate(train_dl):
        b = {k: v.to(device) for k, v in b.items()}
        labels = b.pop("labels")
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            logits = model(**b).logits
        loss = loss_fn(logits.float(), labels)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        running += loss.item()
    train_loss = running / len(train_dl)
    val_probs = get_probs(model, val_dl)
    val_f1 = f1_score(Y_val, val_probs >= 0.5, average="macro", zero_division=0)
    state["epoch"] = epoch + 1
    state["history"].append({"epoch": epoch + 1, "train_loss": train_loss, "val_macro_f1": float(val_f1)})
    print(f"Epoch {epoch+1}/{CFG['epochs']} | train_loss {train_loss:.4f} | val_macro_F1@0.5 {val_f1:.4f} | {time.time()-t0:.0f}s")

    if val_f1 > state["best_f1"]:
        state.update(best_f1=float(val_f1), best_epoch=epoch + 1, bad=0)
        model.save_pretrained(BEST_DIR); tokenizer.save_pretrained(BEST_DIR)
        print("  -> new best model saved")
    else:
        state["bad"] += 1
    torch.save({"model": model.state_dict(), "opt": optimizer.state_dict(), "sched": scheduler.state_dict(),
                "scaler": scaler.state_dict(), "state": state}, CKPT)
    if state["bad"] >= CFG["patience"]:
        print("Early stopping."); break

print("Best epoch:", state["best_epoch"], "val macro-F1:", round(state["best_f1"], 4))

### Cell 7: Validation Threshold Tuning & Test Split Evaluation

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(BEST_DIR).to(device)
val_probs, test_probs = get_probs(model, val_dl), get_probs(model, test_dl)

grid = np.round(np.arange(0.10, 0.91, 0.05), 2)
global_thr = float(max(grid, key=lambda t: f1_score(Y_val, val_probs >= t, average="macro", zero_division=0)))
class_thr = np.array([max(grid, key=lambda t: f1_score(Y_val[:, c], val_probs[:, c] >= t, zero_division=0))
                      for c in range(NUM_LABELS)])

pred_global = test_probs >= global_thr
pred_class  = test_probs >= class_thr
res_global, res_class = compute_metrics(Y_test, pred_global), compute_metrics(Y_test, pred_class)

print(f"Global threshold: {global_thr}")
print(f"{'Strategy':<22}{'macro-F1':>10}{'micro-F1':>10}{'weighted-F1':>13}{'subset-acc':>12}")
for name, r in [("Global threshold", res_global), ("Per-class thresholds", res_class)]:
    print(f"{name:<22}{r['macro']['f1']:>10.4f}{r['micro']['f1']:>10.4f}{r['weighted']['f1']:>13.4f}{r['subset_accuracy']:>12.4f}")

### Cell 8: Error Analysis & Visualizations

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# 1. Top 15 Confused Label Pairs
pair_counts = Counter()
for t_row, p_row in zip(Y_test.astype(bool), pred_class):
    for t in np.where(t_row)[0]:
        for p in np.where(p_row & ~t_row)[0]:
            pair_counts[(LABELS[t], LABELS[p])] += 1
top_pairs = [{"true": a, "predicted": b, "count": n} for (a, b), n in pair_counts.most_common(15)]
print("Top 15 confused pairs (true -> wrongly predicted):")
for x in top_pairs: print(f"  {x['true']:<15} -> {x['predicted']:<15} {x['count']}")

# 2. 5 Worst Classes with Text Examples
worst = sorted(res_class["per_class"].items(), key=lambda kv: kv[1]["f1"])[:5]
worst_examples = {}
print("\n5 worst classes:")
for name, m in worst:
    c = LABELS.index(name)
    missed = [test_texts[i] for i in np.where(Y_test[:, c].astype(bool) & ~pred_class[:, c])[0]][:3]
    worst_examples[name] = {"f1": m["f1"], "support": m["support"], "missed_examples": missed}
    print(f"  {name} (F1 {m['f1']:.3f}, support {m['support']}): {missed}")

# 3. Plots
f1s = {k: v["f1"] for k, v in res_class["per_class"].items()}
order = sorted(f1s, key=f1s.get)
plt.figure(figsize=(8, 8)); plt.barh(order, [f1s[k] for k in order])
plt.xlabel("F1 (test, per-class thresholds)"); plt.title("RoBERTa per-class F1"); plt.tight_layout()
plt.savefig(OUT / "figures" / "roberta_per_class_f1.png", dpi=150); plt.show()

h = state["history"]
plt.figure(figsize=(6, 4))
plt.plot([x["epoch"] for x in h], [x["train_loss"] for x in h], marker="o", label="train loss")
plt.plot([x["epoch"] for x in h], [x["val_macro_f1"] for x in h], marker="o", label="val macro-F1")
plt.xlabel("epoch"); plt.legend(); plt.tight_layout()
plt.savefig(OUT / "figures" / "training_curve.png", dpi=150); plt.show()

pp = top_pairs[:10][::-1]
plt.figure(figsize=(8, 5)); plt.barh([f"{x['true']} -> {x['predicted']}" for x in pp], [x["count"] for x in pp])
plt.title("Most confused label pairs"); plt.tight_layout()
plt.savefig(OUT / "figures" / "confused_pairs.png", dpi=150); plt.show()

### Cell 9: Save Thresholds, Results JSON, and Baseline Comparison

In [ ]:
thresholds = {"labels": LABELS, "global": global_thr,
              "per_class": {LABELS[i]: float(class_thr[i]) for i in range(NUM_LABELS)}}
(BEST_DIR / "thresholds.json").write_text(json.dumps(thresholds, indent=2))
(OUT / "thresholds.json").write_text(json.dumps(thresholds, indent=2))

results = {
    "config": CFG, "best_epoch": state["best_epoch"], "best_val_macro_f1_at_0.5": state["best_f1"],
    "history": state["history"], "global_threshold": global_thr,
    "test_global_threshold": res_global, "test_per_class_thresholds": res_class,
    "top_confused_pairs": top_pairs, "worst_classes": worst_examples,
}
(OUT / "roberta_emotion_results.json").write_text(json.dumps(results, indent=2))
print("Saved results to", OUT)

# Optional: compare with TF-IDF baseline if its JSON is copied to Drive
BASELINE = Path("/content/drive/MyDrive/component3/baseline_emotion_results.json")
if BASELINE.exists():
    b = json.loads(BASELINE.read_text()); b = b.get("test_metrics", b.get("test", b))
    b_macro = b.get("macro_f1", b.get("macro", {}).get("f1", 0.0))
    b_micro = b.get("micro_f1", b.get("micro", {}).get("f1", 0.0))
    b_weighted = b.get("weighted_f1", b.get("weighted", {}).get("f1", 0.0))
    print(f"\n{'Model':<28}{'macro-F1':>10}{'micro-F1':>10}{'weighted-F1':>13}")
    print(f"{'TF-IDF + LR':<28}{b_macro:>10.4f}{b_micro:>10.4f}{b_weighted:>13.4f}")
    for n, r in [("RoBERTa (global thr)", res_global), ("RoBERTa (per-class thr)", res_class)]:
        print(f"{n:<28}{r['macro']['f1']:>10.4f}{r['micro']['f1']:>10.4f}{r['weighted']['f1']:>13.4f}")
else:
    print("\n(Copy your baseline results JSON to", BASELINE, "to print the comparison table.)")

### Cell 10: `predict()` Function & Live Demo

In [ ]:
def predict(texts, apply_thresholds=False):
    """28 probabilities per text. With apply_thresholds=True, also returns predicted labels."""
    model.eval()
    enc = tokenizer(texts, truncation=True, max_length=CFG["max_length"],
                    padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        probs = torch.sigmoid(model(**enc).logits.float()).cpu().numpy()
    out = []
    for row in probs:
        d = {LABELS[i]: float(row[i]) for i in range(NUM_LABELS)}
        if apply_thresholds:
            labels = [LABELS[i] for i in range(NUM_LABELS) if row[i] >= class_thr[i]]
            out.append({"probabilities": d, "predicted_labels": labels})
        else:
            out.append(d)
    return out

# Verification on travel & tourism domain messages
sample_sentences = [
    "I really want to visit Ella!",
    "I don't really like beaches, I'd rather see a historical place.",
    "The hotel is too expensive, I'm worried about the budget."
]

for t in sample_sentences:
    r = predict([t], apply_thresholds=True)[0]
    top3 = sorted(r["probabilities"].items(), key=lambda kv: -kv[1])[:3]
    print(t)
    print("  Active Labels :", r["predicted_labels"])
    print("  Top-3 Probs   :", [(k, round(v, 2)) for k, v in top3])
    print("-" * 70)